# Примеры к Лекции 6: Альтернативные фреймворки для мультиагентных систем

Одна и та же задача — **исследователь и писатель готовят статью** — реализована
на двух фреймворках:

1. **OpenAI Agents SDK** — минимализм, handoffs, guardrails
2. **Pydantic AI** — типобезопасность и структурированный вывод

Оба фреймворка работают через OpenRouter (настройки из `.env`).

> **CrewAI** (часть 3) описан с реальным API, но требует отдельного окружения —
> его зависимость на `pydantic<2.12` конфликтует с Agents SDK и Pydantic AI.
> Инструкции по установке — в секции CrewAI.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

# Проверяем наличие API-ключа
assert os.environ.get("OPENROUTER_API_KEY"), (
    "Установите OPENROUTER_API_KEY в .env файле"
)

# Общая задача для всех фреймворков
TASK = "AI-агенты в 2026 году: ключевые фреймворки, протоколы MCP и A2A, тренды"

# Общая функция поиска (DuckDuckGo, без API-ключа)
from ddgs import DDGS

def ddg_search(query: str, max_results: int = 3) -> str:
    """Поиск через DuckDuckGo — общая функция для всех фреймворков."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=max_results))
    if not results:
        return f"По запросу '{query}' ничего не найдено."
    return "\n".join(
        f"- {r['title']}: {r['body']}" for r in results
    )

# Проверка
print("DuckDuckGo поиск:")
print(ddg_search("AI agent frameworks 2026", max_results=2))

## Часть 1: OpenAI Agents SDK — минимализм и handoffs

Agents SDK определяет агентов минимально: `name`, `instructions`, `handoffs`.
Handoff — передача управления другому агенту с полной историей диалога.

Для работы через OpenRouter используем `OpenAIChatCompletionsModel` —
адаптер для OpenAI-совместимых API.

In [ ]:
# ============================================================================
# OPENAI AGENTS SDK: НАСТРОЙКА МОДЕЛИ ЧЕРЕЗ OPENROUTER
# ============================================================================
from agents import Agent, Runner, function_tool
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI

# OpenRouter как OpenAI-совместимый endpoint
openrouter_client = AsyncOpenAI(
    base_url=os.environ["OPENROUTER_BASE_URL"],
    api_key=os.environ["OPENROUTER_API_KEY"],
)

sdk_model = OpenAIChatCompletionsModel(
    model="openai/gpt-4.1-mini",
    openai_client=openrouter_client,
)

print(f"Agents SDK модель: {sdk_model}")

In [ ]:
# ============================================================================
# AGENTS SDK: ИНСТРУМЕНТЫ
# ============================================================================

@function_tool
def web_search(query: str) -> str:
    """Поиск информации в интернете через DuckDuckGo."""
    return ddg_search(query)

print("Инструмент зарегистрирован:", web_search.name)

In [ ]:
# ============================================================================
# AGENTS SDK: АГЕНТЫ С HANDOFF
# ============================================================================

# Writer определяется первым, потому что Researcher ссылается на него в handoffs
sdk_writer = Agent(
    name="Writer",
    instructions=(
        "Ты — технический писатель. Ты получил результаты исследования "
        "через handoff от исследователя. Напиши статью на 300-400 слов "
        "на русском языке. Структурируй с подзаголовками. "
        "Не выдумывай факты — используй только то, что передал исследователь."
    ),
    model=sdk_model,
)

sdk_researcher = Agent(
    name="Researcher",
    instructions=(
        "Ты — исследователь. Используй инструмент web_search для поиска "
        "актуальных фактов по теме. Найди 5-7 ключевых фактов с цифрами. "
        "Систематизируй результаты. После завершения — передай управление "
        "писателю через handoff."
    ),
    model=sdk_model,
    tools=[web_search],
    handoffs=[sdk_writer],
)

print(f"Researcher: handoffs → {[h.name for h in sdk_researcher.handoffs]}")
print(f"Writer: handoffs → {[h.name for h in sdk_writer.handoffs]}")

In [ ]:
# ============================================================================
# AGENTS SDK: ЗАПУСК
# ============================================================================

# Runner.run_sync() — синхронная обёртка вокруг async Runner.run()
sdk_result = Runner.run_sync(
    starting_agent=sdk_researcher,
    input=f"Исследуй тему и напиши статью: {TASK}",
    max_turns=10,
)

print(f"Финальный агент: {sdk_result.last_agent.name}")
print(f"Кол-во шагов: {len(sdk_result.new_items)}")
print(f"\n--- Результат Agents SDK ---")
print(sdk_result.final_output)

## Часть 2: Pydantic AI — типобезопасность и структурированный вывод

Pydantic AI определяет агентов с явной типизацией: `output_type` гарантирует,
что результат соответствует схеме. Если модель вернула невалидный JSON —
автоматический retry.

Поддержка OpenRouter — встроенная через провайдер `openrouter:`.

In [ ]:
# ============================================================================
# PYDANTIC AI: НАСТРОЙКА
# ============================================================================
from pydantic_ai import Agent as PydanticAgent
from pydantic import BaseModel

# Pydantic AI имеет встроенную поддержку OpenRouter
# Формат модели: "openrouter:<provider>/<model>"
# Требует env var OPENROUTER_API_KEY (уже загружен через dotenv)
PYDANTIC_MODEL = "openrouter:openai/gpt-4.1-mini"

print(f"Pydantic AI модель: {PYDANTIC_MODEL}")

In [ ]:
# ============================================================================
# PYDANTIC AI: ТИПИЗИРОВАННЫЙ ИССЛЕДОВАТЕЛЬ
# ============================================================================

# Структурированный результат — Pydantic AI валидирует автоматически
class ResearchResult(BaseModel):
    """Результат исследования — типизированная схема."""
    facts: list[str]
    sources_count: int
    main_trend: str


# Агент с типизированным выводом
researcher = PydanticAgent(
    model=PYDANTIC_MODEL,
    output_type=ResearchResult,
    system_prompt=(
        "Ты — исследователь. Используй инструмент research_search для поиска "
        "актуальных фактов по теме. Верни результат в структурированном формате: "
        "список фактов, количество источников, главный тренд."
    ),
)

# Инструмент привязывается к агенту через декоратор
@researcher.tool_plain
def research_search(query: str) -> str:
    """Поиск информации в интернете через DuckDuckGo."""
    return ddg_search(query)


print(f"Researcher: output_type={researcher.output_type.__name__}")

In [ ]:
# ============================================================================
# PYDANTIC AI: ПИСАТЕЛЬ + ЗАПУСК ЦЕПОЧКИ
# ============================================================================

# Писатель — простой строковый вывод
writer = PydanticAgent(
    model=PYDANTIC_MODEL,
    output_type=str,
    system_prompt=(
        "Ты — технический писатель. Напиши статью на 300-400 слов "
        "на русском языке на основе предоставленных фактов. "
        "Структурируй с подзаголовками. Не выдумывай факты."
    ),
)

# Запуск цепочки: исследователь → писатель
# В Pydantic AI нет встроенной multi-agent оркестрации — цепочка через код
print("Шаг 1: Исследование (типизированный результат)")
research_result = researcher.run_sync(f"Исследуй тему: {TASK}")

# Доступ к типизированному результату
data = research_result.output
print(f"  Тип: {type(data).__name__}")
print(f"  Фактов: {len(data.facts)}")
print(f"  Источников: {data.sources_count}")
print(f"  Главный тренд: {data.main_trend}")
for i, fact in enumerate(data.facts, 1):
    print(f"  {i}. {fact[:80]}...")

# Передаём факты писателю
print("\nШаг 2: Написание статьи")
facts_text = "\n".join(f"- {f}" for f in data.facts)
article_result = writer.run_sync(
    f"Напиши статью на основе фактов:\n{facts_text}\nГлавный тренд: {data.main_trend}"
)

print(f"\n--- Результат Pydantic AI ---")
print(article_result.output)

## Часть 3: CrewAI — роли, задачи и экипаж

CrewAI определяет агентов через `role`, `goal`, `backstory` — три строки
на естественном языке. Задача — через `description` и `expected_output`.
Экипаж (`Crew`) — список агентов и задач. `crew.kickoff()` запускает выполнение.

> **Установка:** CrewAI конфликтует с Agents SDK и Pydantic AI по версии pydantic
> (CrewAI требует `<2.12`, остальные — `>=2.12`). Для запуска этой секции
> создайте отдельное окружение:
> ```bash
> uv venv .venv-crewai
> source .venv-crewai/bin/activate
> uv pip install crewai duckduckgo-search python-dotenv
> ```

In [ ]:
# ============================================================================
# CREWAI: РЕАЛЬНЫЙ КОД (требует отдельного окружения)
# ============================================================================
try:
    from crewai import Agent as CrewAgent, Task, Crew, LLM
    from crewai.tools import tool as crewai_tool
    CREWAI_AVAILABLE = True
except ImportError:
    CREWAI_AVAILABLE = False
    print("CrewAI не установлен в текущем окружении.")
    print("Для установки: uv venv .venv-crewai && uv pip install crewai duckduckgo-search python-dotenv")
    print("\nНиже — код, который выполнится при наличии CrewAI:")

if CREWAI_AVAILABLE:
    # LLM через OpenRouter (CrewAI использует LiteLLM внутри)
    crewai_llm = LLM(
        model="openrouter/openai/gpt-4.1-mini",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )

    # Инструмент поиска — CrewAI имеет свой декоратор @tool
    @crewai_tool("Поиск в интернете")
    def crewai_search(query: str) -> str:
        """Поиск информации в интернете через DuckDuckGo."""
        return ddg_search(query)

    # Агенты — определяются через роль, цель и предысторию
    crew_researcher = CrewAgent(
        role="Старший исследователь технологий",
        goal="Найти и систематизировать актуальные факты по заданной теме",
        backstory=(
            "Ты работаешь в аналитическом отделе технологической компании. "
            "10 лет опыта в исследовании IT-трендов. Ценишь точные данные, "
            "цифры и конкретные примеры."
        ),
        tools=[crewai_search],
        llm=crewai_llm,
        verbose=True,
    )

    crew_writer = CrewAgent(
        role="Технический писатель",
        goal="Написать увлекательную и структурированную статью на основе фактов",
        backstory=(
            "Ты — автор технологического блога с 50 000 подписчиков. "
            "Умеешь объяснять сложные вещи простым языком. "
            "Никогда не выдумываешь факты."
        ),
        llm=crewai_llm,
        verbose=True,
    )

    # Задачи
    research_task = Task(
        description=f"Исследуй тему: {TASK}. Найди 5-7 ключевых фактов с цифрами.",
        expected_output="Структурированный список из 5-7 ключевых фактов",
        agent=crew_researcher,
    )

    writing_task = Task(
        description="Напиши статью на 300-400 слов на русском языке на основе собранных фактов.",
        expected_output="Готовая статья с заголовком и 3-4 разделами",
        agent=crew_writer,
    )

    # Экипаж — запуск
    crew = Crew(
        agents=[crew_researcher, crew_writer],
        tasks=[research_task, writing_task],
        verbose=True,
    )

    crew_result = crew.kickoff()
    print(f"\n--- Результат CrewAI ---")
    print(crew_result)
else:
    print("""
# Код CrewAI (для справки):

from crewai import Agent, Task, Crew, LLM

llm = LLM(model="openrouter/openai/gpt-4.1-mini", api_key=os.environ["OPENROUTER_API_KEY"])

researcher = Agent(
    role="Старший исследователь технологий",
    goal="Найти факты по теме",
    backstory="10 лет опыта в IT-аналитике",
    tools=[crewai_search],
    llm=llm,
)

writer = Agent(
    role="Технический писатель",
    goal="Написать статью на основе фактов",
    backstory="Автор блога с 50k подписчиков",
    llm=llm,
)

task1 = Task(description="Исследуй тему...", expected_output="Факты", agent=researcher)
task2 = Task(description="Напиши статью...", expected_output="Статья", agent=writer)

crew = Crew(agents=[researcher, writer], tasks=[task1, task2])
result = crew.kickoff()
""")

## Часть 4: Сравнение подходов

Три фреймворка — три философии решения одной задачи.

In [ ]:
# ============================================================================
# СРАВНЕНИЕ
# ============================================================================

comparison = {
    "CrewAI": {
        "Определение агента": "role + goal + backstory (естественный язык)",
        "Координация": "Crew автоматически передаёт контекст между задачами",
        "Инструменты": "@crewai_tool декоратор или LangChain tools",
        "Модели": "LiteLLM (OpenRouter, OpenAI, Anthropic, 100+ провайдеров)",
        "Кому подходит": "Задачи через роли; понятность для нетехнических участников",
        "Ограничение": "pydantic<2.12 — конфликт с Agents SDK и Pydantic AI",
    },
    "Agents SDK": {
        "Определение агента": "name + instructions + handoffs (минимализм)",
        "Координация": "Handoff — передача эстафеты с полной историей",
        "Инструменты": "@function_tool декоратор",
        "Модели": "OpenAI по умолчанию, OpenRouter через ChatCompletionsModel",
        "Кому подходит": "Линейные пайплайны, routing, customer support",
        "Ограничение": "Только handoffs; нет графов, fan-out, shared state",
    },
    "Pydantic AI": {
        "Определение агента": "model + system_prompt + output_type (типизированно)",
        "Координация": "Ручная через код — фреймворк не навязывает паттерн",
        "Инструменты": "@agent.tool_plain декоратор",
        "Модели": "Встроенная поддержка OpenRouter, OpenAI, Anthropic, Google и др.",
        "Кому подходит": "Production-код, строгая типизация, durable execution",
        "Ограничение": "Нет встроенной multi-agent оркестрации",
    },
}

for framework, props in comparison.items():
    print(f"\n{'='*60}")
    print(f"  {framework}")
    print(f"{'='*60}")
    for key, value in props.items():
        print(f"  {key}:")
        print(f"    {value}")

print("\n" + "="*60)
print("  ВЫВОД")
print("="*60)
print("  Паттерны (supervisor, handoffs, reflection) — универсальны.")
print("  Фреймворки — это реализации. Изучайте паттерны глубоко,")
print("  фреймворки — широко.")